# 实验3：CNN图像识别与路标分类
## 2.2 数据预处理与数据增强

In [ ]:
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import os

# 定义标准预处理（用于验证集和最终测试）
data_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.3337, 0.3064, 0.3171), (0.2672, 0.2564, 0.2629))
])

# 定义带亮度抖动的数据增强（仅用于训练）
data_jitter_brightness = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ColorJitter(brightness=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.3337, 0.3064, 0.3171), (0.2672, 0.2564, 0.2629))
])

print("数据预处理与增强管道定义完成")

### 数据增强展示
（请确保以下图片路径存在，如果不存在请修改为数据集中任意一张图片的路径）

In [ ]:
# 尝试加载一张训练图片（如果路径不存在请自行修改）
img_path = "./data/train_images/0/00000_00003_00000.png"
if not os.path.exists(img_path):
    # 如果默认路径不存在，尝试搜索任意一个 png 文件
    import glob
    candidates = glob.glob("./data/train_images/*/*.png")
    if candidates:
        img_path = candidates[0]
        print(f"使用图片: {img_path}")
    else:
        print("未找到图片，请手动指定一张图片路径。")
        img_path = None

if img_path and os.path.exists(img_path):
    original_img = Image.open(img_path).convert('RGB')
    
    # 应用增强变换
    augmented_img_tensor = data_jitter_brightness(original_img)
    augmented_img = transforms.ToPILImage()(augmented_img_tensor)
    
    # 展示原图和增强后的图
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title("Original Image")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(augmented_img)
    plt.title("Brightness Augmented")
    plt.axis('off')
    plt.show()
else:
    print("无法展示数据增强，请检查图片路径。")

## 2.3 定义网络架构

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ImprovedNet(nn.Module):
    """改进的CNN网络，包含4个卷积层+BN+Dropout，用于GTSRB分类"""
    def __init__(self, nclasses=43):
        super(ImprovedNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        # 经过4次池化后特征图大小：32 -> 16 -> 8 -> 4 -> 2
        self.fc1 = nn.Linear(512 * 2 * 2, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, nclasses)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(-1, 512 * 2 * 2)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

# 实例化模型并打印结构（用于截图）
model = ImprovedNet()
print(model)